<a href="https://colab.research.google.com/github/hargagan/Machine-Learning/blob/master/Breast_cancer_Dataset_classification_asgnmnt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt; import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler  # Scaling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import log_loss, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
# from sklearn.metrics import classification_report, roc_curve, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import label_binarize
import warnings; warnings.filterwarnings('ignore')

In [30]:
# Load the breast cancer data
breast_cancer =  load_breast_cancer()
X = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
y = pd.Series(breast_cancer.target)

print(f'Input shape: {X.shape}')
print(f'Class distribution:')
for i, class_name in enumerate(breast_cancer.target_names):
    count = (y == i).sum()
    print(f'  {class_name}: {count} ({count/len(y):.1%})')

Input shape: (569, 30)
Class distribution:
  malignant: 212 (37.3%)
  benign: 357 (62.7%)


In [20]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, stratify=y, random_state=9001)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")

Train: 455, Test: 114
Train class distribution: [170 285]
Test class distribution: [42 72]


In [22]:
# LogisticRegression?

Remember, you should fit the scaler only on the training data and then apply that fitted transformation to the testing data. This ensures the testing set remains unseen and unbiased, while still being scaled consistently with the training data.

In [23]:
# Fit a scaler to the training data and transform all the datasets
scaler = MinMaxScaler()
numerical_columns = X_train.select_dtypes(include = ['int64', 'float64']).columns
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [24]:
model = LogisticRegression(random_state = 9001, multi_class='ovr')  # One-vs-Rest
model.fit(X_train, y_train)
print(f'Intercept: {model.intercept_}')
print(f'Coefficients shape: {model.coef_.shape} (1 classes, 30 features)')
print(f'Classes: {model.classes_}')

Intercept: [8.9697839]
Coefficients shape: (1, 30) (1 classes, 30 features)
Classes: [0 1]


In [38]:
pd.DataFrame({"Feature": X_train.columns, "Coefficients - LR": model.coef_[0]}).round(2)

,Feature,Coefficients - LR
0,mean radius,-1.74
1,mean texture,-1.49
2,mean perimeter,-1.71
3,mean area,-1.47
4,mean smoothness,-0.35
5,mean compactness,-0.46
6,mean concavity,-1.51
7,mean concave points,-1.78
8,mean symmetry,-0.46
9,mean fractal dimension,0.85


In [25]:
y_proba = model.predict_proba(X_test)  # Probabilities
y_pred = model.predict(X_test)  # Predictions
results_df = pd.DataFrame({'Actual': y_test.iloc[:5].values,'Prob_Die': y_proba[:5, 0],'Prob_Survive': y_proba[:5, 1],'Prediction': y_pred[:5]})
print(results_df.round(3))

   Actual  Prob_Die  Prob_Survive  Prediction
0       1     0.194         0.806           1
1       0     0.996         0.004           0
2       1     0.057         0.943           1
3       1     0.005         0.995           1
4       0     0.949         0.051           0


In [26]:
print(classification_report(y_test, y_pred, target_names = ['malignant', 'benign']))

              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

